<a href="https://colab.research.google.com/github/doa-2026/machine-learning/blob/main/EnsembleTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load Data

In [6]:
path="/content/drive/MyDrive/Boston_Housing_from_Sklearn - Boston_Housing_from_Sklearn.csv"
import pandas as pd
df=pd.read_csv(path)
df.head()

,CRIM,NOX,RM,AGE,PTRATIO,LSTAT,PRICE
0,0.00632,0.538,6.575,65.2,15.3,4.98,24.0
1,0.02731,0.469,6.421,78.9,17.8,9.14,21.6
2,0.02729,0.469,7.185,61.1,17.8,4.03,34.7
3,0.03237,0.458,6.998,45.8,18.7,2.94,33.4
4,0.06905,0.458,7.147,54.2,18.7,5.33,36.2


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


import packages

In [8]:
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.model_selection import train_test_split ,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score , mean_absolute_error , mean_squared_error
from sklearn.ensemble import BaggingRegressor ,RandomForestRegressor



perform validation split

In [9]:
X=df.drop(columns=["PRICE"])
y=df["PRICE"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

Train Bagging Tree

In [10]:
bag=BaggingRegressor(random_state=42)
bag.fit(X_train,y_train)


BaggingRegressor(random_state=42)

show the hyperparameters

In [19]:
bag.get_params()

{'bootstrap': True,
 'bootstrap_features': False,
 'estimator': None,
 'max_features': 1.0,
 'max_samples': 1.0,
 'n_estimators': 10,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [11]:
# Add custom function

def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

Tune the model With GridsearchCV

In [44]:
param_grid={"n_estimators":[5,4,10]
            ,'max_samples':[0.5,.7,1.0]
            ,'max_features':[.5,.7,.9]}
grid=GridSearchCV(bag,param_grid,n_jobs=-1 ,verbose=1)
grid.fit(X_train,y_train)
grid.best_params_
bestbag=grid.best_estimator_

Fitting 5 folds for each of 27 candidates, totalling 135 fits


Evalute the model

In [45]:
evaluate_regression(bestbag, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.227
- MSE = 3.790
- R^2 = 0.957

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.560
- MSE = 15.681
- R^2 = 0.776


Train the model

In [30]:
rf=RandomForestRegressor(random_state=42)
rf.fit(X_train,y_train)
rf.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

tune with GridSearchCV

In [46]:
param_grid={"n_estimators":[5,10]
            , 'max_samples':[0.5,1.0]
            ,'max_features':[.5,.7,.9]}

Grid=GridSearchCV(rf,param_grid,n_jobs=-1 ,verbose=1)
Grid.fit(X_train,y_train)
Grid.best_params_
bestfor=Grid.best_estimator_

Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [41]:
evaluate_regression(Grid, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.092
- MSE = 3.216
- R^2 = 0.964

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.249
- MSE = 12.393
- R^2 = 0.823


Evalute the model

In [47]:
evaluate_regression(bestfor, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.092
- MSE = 3.216
- R^2 = 0.964

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 2.249
- MSE = 12.393
- R^2 = 0.823


the testing R2 score improved from .776 to .823 by using random forest ...
so random forest algorithm can predict the target better than bagging and close to true value